In [1]:
import pandas as pd
df = pd.read_csv('books_full_with_description.csv')

In [2]:
df.sample(10)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,description,genres_raw,match_method
4867,590431307,The Forgotten Door,Alexander Key,1989,Scholastic,Far from home.Jon has lost his memory. He can'...,"['Science Fiction', 'Fantasy', 'Young Adult', ...",isbn
6667,60508213,Traci Lords: Underneath It All,Traci Lords,2004,Perennial Currents,"The moving, gripping, and tell–all autobiograp...","['Biography', 'Nonfiction', 'Memoir', 'Autobio...",isbn
1743,60914440,Skywriting by Word of Mouth : And Other Writin...,John Lennon,1987,Perennial,John Lennon wrote Skywriting by Word of Mouth...,"['Music', 'Poetry', 'Fiction', 'Humor', 'Short...",isbn
12706,671688359,IF YOU REALLY LOVED ME,Ann Rule,1991,Simon &amp; Schuster,There was only one way to please her father: ...,"['True Crime', 'Nonfiction', 'Crime', 'Mystery...",title_author
13380,701118822,Bring Me a Unicorn: Diaries and Letters of Ann...,Anne Morrow Lindbergh,1976,Vintage/Ebury (A Division of Random House Group),The first volume of Lindbergh’s diaries and le...,"['Nonfiction', 'Biography', 'Memoir', 'History...",title_author
3233,067102082X,The Gun Seller,Hugh Laurie,1998,Washington Square Press,"When Thomas Lang, a hired gunman with a soft h...","['Fiction', 'Humor', 'Mystery', 'Thriller', 'C...",isbn
1729,684818442,FROM TIME TO TIME,Jack Finney,1996,Touchstone,"Jack Finney's beloved sequel to his classic, N...","['Time Travel', 'Fiction', 'Science Fiction', ...",isbn
13531,1853260797,The Taming of the Shrew,William Shakespeare,1999,Wordsworth Editions Ltd,Renowned as Shakespeare's most boisterous come...,"['Classics', 'Plays', 'Fiction', 'Drama', 'Sch...",title_author
11699,553207229,Dragondrums,Anne McCaffrey,1979,Bantam Books,Dragondrums is the coming of age story of Piem...,"['Fantasy', 'Science Fiction', 'Dragons', 'Fic...",title_author
13523,192836714,The Master of Ballantrae,Robert Louis Stevenson,1999,Oxford University Press,Set in Scotland during the 1745 Jacobite Rebel...,"['Classics', 'Fiction', 'Historical Fiction', ...",title_author


**Content Based Filtering Using the Vector Embeddings**

In [4]:
import pandas as pd
import numpy as np
import re, ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

# ============================================
# PART 1: CONTENT-BASED SETUP
# ============================================
books = pd.read_csv('books_full_with_description.csv')

def parse_genres(g):
    try:
        v = ast.literal_eval(g)
        return v if isinstance(v, list) else []
    except Exception:
        return []

books['genres_list'] = books['genres_raw'].apply(parse_genres)
books['genres_text'] = books['genres_list'].apply(
    lambda gs: ' '.join(str(g).replace(' ', '_') for g in gs))

books['dedupe_key'] = (books['Book-Title'].str.lower().str.strip() + '||' +
                       books['Book-Author'].astype(str).str.lower().str.strip())
books = books.drop_duplicates(subset='dedupe_key').reset_index(drop=True)

def clean_text(t):
    t = str(t).lower()
    t = re.sub(r'[^a-z\s_]', ' ', t)
    return ' '.join(t.split())

def build_soup(title, description, genres_text, title_w=1, genre_w=3):
    return ' '.join([clean_text(title)] * title_w +
                    [clean_text(genres_text)] * genre_w +
                    [clean_text(description)])





















books['soup'] = [build_soup(t, d, g) for t, d, g in
                 zip(books['Book-Title'], books['description'], books['genres_text'])]
books = books[books['soup'].str.strip() != ''].reset_index(drop=True)

vectorizer = TfidfVectorizer(stop_words='english', max_features=30000,
                             ngram_range=(1, 2), min_df=2)
content_matrix = normalize(vectorizer.fit_transform(books['soup']))

title_to_idx = pd.Series(books.index, index=books['Book-Title'].str.lower().str.strip())
title_to_idx = title_to_idx[~title_to_idx.index.duplicated(keep='first')]

print("Content matrix:", content_matrix.shape)


Content matrix: (11013, 30000)


In [5]:
print(content_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 833425 stored elements and shape (11013, 30000)>
  Coords	Values
  (0, 26721)	0.06063428639760238
  (0, 10116)	0.03798081931555882
  (0, 27080)	0.09317210231025762
  (0, 17908)	0.07981262791357285
  (0, 15305)	0.21039257124675925
  (0, 26175)	0.10417865016654924
  (0, 6557)	0.0956844265391296
  (0, 18012)	0.10187954169323504
  (0, 15183)	0.1532738165020067
  (0, 366)	0.11105346319727988
  (0, 19006)	0.06442060523768445
  (0, 28560)	0.057953332589164706
  (0, 19335)	0.057637442429754944
  (0, 22702)	0.04307038164734367
  (0, 1362)	0.059574885385480995
  (0, 19357)	0.030759580848492965
  (0, 16562)	0.02705440045398141
  (0, 7137)	0.0660962398100621
  (0, 14677)	0.0675492619131999
  (0, 13217)	0.05484899431125035
  (0, 2359)	0.08205663800984607
  (0, 27684)	0.1502964362446998
  (0, 20222)	0.2623598760544672
  (0, 28748)	0.04582600339307568
  (0, 24160)	0.06135798063801752
  :	:
  (11012, 4166)	0.06394879499052937
  (11012, 1188

In [6]:
# title lookup
title_to_idx = pd.Series(books.index, index=books['Book-Title'].str.lower().str.strip())
title_to_idx = title_to_idx[~title_to_idx.index.duplicated(keep='first')]

# ---------- 4. Helper: top-k results ----------
def _top_k(sim, k, exclude_idx=None):
    if exclude_idx is not None:
        sim[exclude_idx] = -1
    idx = np.argpartition(-sim, min(k, len(sim) - 1))[:k]
    idx = idx[np.argsort(-sim[idx])]
    out = books.iloc[idx][['Book-Title', 'Book-Author']].copy()
    out['score'] = sim[idx].round(3)
    if 'genres_list' in books.columns:
        out['genres'] = [', '.join(g[:4]) if isinstance(g, list) else str(g) for g in books.iloc[idx]['genres_list']]
    elif 'genres_raw' in books.columns:
        out['genres'] = books.iloc[idx]['genres_raw']
    else:
        out['genres'] = ''
    return out.reset_index(drop=True)


In [7]:
# ---------- 5A. Recommend for a book IN the dataset ----------
def content_recommend(title, k=5):
    key = str(title).lower().strip()
    if key not in title_to_idx:
        return None
    i = title_to_idx[key]
    sim = (content_matrix @ content_matrix[i].T).toarray().ravel()
    return _top_k(sim, k, exclude_idx=i)

In [8]:
# ---------- 5B. Recommend for a NEW book (not in dataset) ----------
# Key point: use vectorizer.transform(), NOT fit_transform() —
# the new book must be projected into the SAME vocabulary space.
def recommend_new_book(title, description='', genres=None, k=5):
    gtext = ' '.join(str(g).replace(' ', '_') for g in (genres or []))
    soup = build_soup(title, description, gtext)
    new_vec = normalize(vectorizer.transform([soup]))
    sim = (content_matrix @ new_vec.T).toarray().ravel()
    return _top_k(sim, k)

In [9]:
# ---------- 6. Unified entry point ----------
def recommend(title, description=None, genres=None, k=5):
    """Tries the dataset first; falls back to vectorizing provided info."""
    result = content_recommend(title, k)
    if result is not None:
        print(f"[found in dataset] {title}")
        return result
    print(f"[new book] '{title}' not in dataset — using provided description/genres")
    return recommend_new_book(title, description or '', genres or [], k)

In [10]:
# ---------- Examples ----------
print(recommend("The Testaments", k=5))


[new book] 'The Testaments' not in dataset -> using provided description/genres
                   Book-Title              Book-Author  score                                        genres
0                  Silverhill             Phyllis A. Whitney  0.407   Mystery, Gothic, Romance, Romantic Suspense
1                   Cathedral               Raymond Carver  0.375  Short Stories, Fiction, Classics, Literature
2  A spy in the house of love                  Anas Nin  0.364           Fiction, Erotica, Classics, Romance
3         Cities of the Plain            Cormac McCarthy  0.354         Fiction, Westerns, Literature, Novels
4                   The Fever                 Wallace Shawn  0.342                Plays, Fiction, Drama, Theatre


In [11]:
print(recommend(
    "The Psychology of Money",
    description=("Timeless lessons on wealth, greed, and happiness. Doing well with money isn't necessarily "
                 "about what you know. It's about how you behave. And behavior is hard to teach, even to really "
                 "smart people. How to manage money, invest it, and make personal finance and business decisions."),
    genres=["Nonfiction", "Finance", "Business", "Economics", "Money", "Psychology", "Self Help"],
    k=5))

[new book] 'The Psychology of Money' not in dataset -> using provided description/genres
                                                                                                Book-Title          Book-Author  score                                            genres
0                                            The Total Money Makeover: A Proven Plan for Financial Fitness          Dave Ramsey  0.389  Nonfiction, Finance, Self Help, Personal Finance
1  Your Money or Your Life: Transforming Your Relationship With Money and Achieving Financial Independence        Joe Dominguez  0.383      Finance, Nonfiction, Personal Finance, Money
2                                                                                       Beating the Street          Peter Lynch  0.371          Finance, Business, Nonfiction, Economics
3                                                                               The Richest Man in Babylon     George S. Clason  0.367          Finance, Business, Self Hel

**Collaborative Filtering**


In [13]:
# ============================================
# Collaborative Filtering — Item-Item Cosine Similarity
# Upload ratings_cleaned.csv and books_full_with_description.csv to Colab first
# ============================================

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load data
ratings = pd.read_csv('ratings_cleaned.csv')
raw_books = pd.read_csv('books_full_with_description.csv')

# 2. Bring in book titles for readability (keeping all ratings, including 0s)
merged = ratings.merge(raw_books[['ISBN', 'Book-Title']], on='ISBN', how='inner')

# 3. Apply thresholds: books with >50 ratings, users who rated >25 books
book_rating_counts = merged.groupby('Book-Title')['Book-Rating'].count()
user_rating_counts = merged.groupby('User-ID')['Book-Rating'].count()

popular_books = book_rating_counts[book_rating_counts > 50].index
active_users = user_rating_counts[user_rating_counts > 25].index

filtered = merged[
    merged['Book-Title'].isin(popular_books) &
    merged['User-ID'].isin(active_users)
]

print(f"Filtered ratings: {len(filtered)}")
print(f"Books: {filtered['Book-Title'].nunique()}, Users: {filtered['User-ID'].nunique()}")

# 4. Build the pivot table: rows = books, columns = users, values = rating
#    Missing entries (user never interacted with that book) -> 0
pivot = filtered.pivot_table(
    index='Book-Title',
    columns='User-ID',
    values='Book-Rating'
).fillna(0)

print("Pivot table shape:", pivot.shape)

# 5. Compute item-item cosine similarity (book x book matrix)
cf_similarity = cosine_similarity(pivot)
cf_similarity_df = pd.DataFrame(cf_similarity, index=pivot.index, columns=pivot.index)

# 6. Function: get top-k similar books by collaborative filtering
def cf_recommend(book_title, k=5):
    if book_title not in cf_similarity_df.index:
        return f"'{book_title}' not in the filtered CF matrix (needs >50 ratings)."
    scores = cf_similarity_df[book_title].drop(book_title)
    top_k = scores.sort_values(ascending=False).head(k)
    return top_k

# Example
print(cf_recommend(pivot.index[5], k=10))


Filtered ratings: 39434
Books: 631, Users: 1112
Pivot table shape: (631, 1112)
Book-Title
The Lost Boy: A Foster Child's Search for the Love of a Family                         0.309283
A Girl Named Zippy: Growing Up Small in Mooreland Indiana (Today Show Book Club #3)    0.235712
Family Album                                                                           0.200944
Gap Creek: The Story Of A Marriage                                                     0.198783
Never Change                                                                           0.173438
Open House (Oprah's Book Club (Paperback))                                             0.170238
She's Come Undone (Oprah's Book Club)                                                  0.161520
I Know Why the Caged Bird Sings                                                        0.159623
A Man Named Dave: A Story of Triumph and Forgiveness                                   0.157609
Where the Heart Is (Oprah's Book Club (Paperba

In [14]:
# ============================================
# WEIGHTED HYBRID RECOMMENDER
# 70% Collaborative Filtering + 30% Content-Based
# ============================================


# ============================================
# PART 2: COLLABORATIVE FILTERING SETUP
# ============================================
ratings = pd.read_csv('ratings_cleaned.csv')
raw_books = pd.read_csv('books_full_with_description.csv')

merged = ratings.merge(raw_books[['ISBN', 'Book-Title']], on='ISBN', how='inner')

book_counts = merged.groupby('Book-Title')['Book-Rating'].count()
user_counts = merged.groupby('User-ID')['Book-Rating'].count()
popular_books = book_counts[book_counts > 50].index
active_users = user_counts[user_counts > 25].index

filtered = merged[merged['Book-Title'].isin(popular_books) &
                  merged['User-ID'].isin(active_users)]

pivot = filtered.pivot_table(index='Book-Title', columns='User-ID',
                             values='Book-Rating').fillna(0)
cf_similarity = cosine_similarity(pivot)
cf_df = pd.DataFrame(cf_similarity, index=pivot.index, columns=pivot.index)

print("CF pivot table:", pivot.shape)

# The 'books' DataFrame from the content-based setup (cell 'a7EXNxzQ5BJe')
# is used as the reference here, which has 11013 rows and is consistent
# with content_matrix and title_to_idx. The explicit reloading of 'books'
# in this cell is removed to ensure this consistency.

# The 'title_positions' variable is no longer needed as 'title_to_idx'
# serves the same purpose and is correctly aligned with content_matrix.


# ============================================
# PART 3: WEIGHTED HYBRID
# ============================================
ALPHA = 0.7   # 0.7 = 70% collaborative filtering, 30% content-based

def hybrid_recommend(title, k=5, alpha=ALPHA, description=None, genres=None):
    """
    final_score = alpha * CF_score + (1 - alpha) * content_score
    Falls back to pure content if the book has no rating data.
    """
    key = str(title).lower().strip()

    # n should be the number of books in the content-filtered dataset
    # (which is the size of content_matrix's first dimension)
    n = content_matrix.shape[0] # Ensure consistency with content_scores (11013)

    # ---- Content score ----
    if key in title_to_idx:
        i = title_to_idx[key]
        content_scores = (content_matrix @ content_matrix[i].T).toarray().ravel()
        content_scores[i] = -1          # exclude the book itself
    else:
        gtext = ' '.join(str(g).replace(' ', '_') for g in (genres or []))
        soup = build_soup(title, description or '', gtext)
        new_vec = normalize(vectorizer.transform([soup]))
        content_scores = (content_matrix @ new_vec.T).toarray().ravel()

    # ---- CF score (aligned to the same book order) ----
    cf_scores = np.zeros(n) # Initialize cf_scores with the correct length 'n' (11013)
    cf_available = title in cf_df.index

    if cf_available:
        row = cf_df[title]
        # Use title_to_idx to map CF book titles to content-filtered book indices
        common_titles_in_content = row.index.intersection(title_to_idx.index)

        # Get the indices in the 'books' DataFrame (content-filtered) for these common titles
        mapped_indices = title_to_idx[common_titles_in_content].values

        # Assign CF scores to the correct positions in cf_scores
        cf_scores[mapped_indices] = row[common_titles_in_content].values

        # Exclude the book itself if it's in the content-filtered books
        if key in title_to_idx: # Use 'key' for consistency with title_to_idx
            cf_scores[title_to_idx[key]] = -1   # exclude itself

    # ---- Blend ----
    effective_alpha = alpha if cf_available else 0.0
    final_scores = effective_alpha * cf_scores + (1 - effective_alpha) * content_scores

    # The 'books' DataFrame here refers to the content-filtered one (11013 rows)
    idx = np.argsort(-final_scores)[:k]
    out = books.iloc[idx][['Book-Title', 'Book-Author']].copy()
    out['final_score']   = final_scores[idx].round(3)
    out['cf_score']      = cf_scores[idx].round(3)
    out['content_score'] = content_scores[idx].round(3)

    mode = f"CF {int(effective_alpha*100)}% + Content {int((1-effective_alpha)*100)}%" \
           if cf_available else "Content only (no rating data)"
    print(f"Recommendations for '{title}'  [{mode}]")
    return out.reset_index(drop=True)


# ============================================
# EXAMPLES
# ============================================
print(hybrid_recommend("The Testament", k=5))

print(hybrid_recommend(
    "The Silent Patient", k=5,
    description="A psychotherapist becomes obsessed with a painter who shot her husband and stopped speaking. A dark psychological thriller.",
    genres=["Thriller", "Mystery", "Psychological Thriller"]))

CF pivot table: (631, 1112)
Recommendations for 'The Testament'  [CF 70% + Content 30%]
                           Book-Title       Book-Author  final_score  \
0  Robin Hood (Wordsworth Collection)     Henry Gilbert        0.103   
1                            The Firm      John Grisham        0.102   
2                              Zodiac  Robert Graysmith        0.092   
3                    The Tin Princess    PHILIP PULLMAN        0.092   
4                The Bourne Ultimatum     Robert Ludlum        0.080   

   cf_score  content_score  
0       0.0          0.342  
1       0.0          0.340  
2       0.0          0.307  
3       0.0          0.307  
4       0.0          0.267  
Recommendations for 'The Silent Patient'  [Content only (no rating data)]
                      Book-Title            Book-Author  final_score  \
0               Beneath the Skin           Nicci French        0.320   
1  House of Glass (Buru Quartet)  Pramoedya Ananta Toer        0.302   
2              

In [15]:
print(hybrid_recommend("The Catcher in the Rye", k=5))

Recommendations for 'The Catcher in the Rye'  [CF 70% + Content 30%]
                                          Book-Title              Book-Author  \
0  The Last Report on the Miracles at Little No H...           Louise Erdrich   
1                                      No Safe Place  RICHARD NORTH PATTERSON   
2                        Rule of the Bone : Novel, A            Russell Banks   
3                                The Marching Season             Daniel Silva   
4  The Ultimate Egoist: The Complete Stories of T...        Theodore Sturgeon   

   final_score  cf_score  content_score  
0        0.300     0.000          1.000  
1        0.162     0.197          0.082  
2        0.068     0.000          0.226  
3        0.067     0.000          0.223  
4        0.051     0.000          0.171  
